# Utils y librerías




In [ ]:
!pip install unidecode

In [ ]:
!pip install thefuzz

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# UTILS
from unidecode import unidecode
from thefuzz import process
from nltk.tokenize import word_tokenize
from openai import OpenAI
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from itertools import chain
from collections import Counter



import unicodedata
import torch
import nltk
import re
import string
import textwrap
import ast


In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


In [ ]:
pd.set_option('display.max_rows', None)

# 1.Caracterización

In [ ]:
char_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-characterization-file-completed.csv'


In [ ]:
char_file = pd.read_csv(char_file_path)


In [ ]:
char_file.columns

El archivo de caracterización cuenta con columnas de identificación de la persona:
- username
- email
- name
- uncode_usr
- ocupación
- otra ocupación adicional
- platform

y también cuenta con columnas informativas relacionadas al curso:

- dónde conoció el curso
- motivación
- lenguajes de programación
- nivel de programación
- dedicación semanal
- razón para realizar el curso
- otros medios por los que se enteró del curso
- otras motivaciones para iniciar el curso.
- otro lenguaje de programación diferentes a los de la lista

La columna 'encuesta inicial' puede quitarse pues es un json que contiene la información de las demás columnas.

In [ ]:
char_file['encuestainicial'][0]

In [ ]:
char_file.drop(columns=['encuestainicial'], inplace=True)

In [ ]:
len(char_file)

## ocupación y otra_ocupacion

esta variable se llenó a partir de un formulario cerrado. Sin embargo, al momento de unir los openedx con edunext, los textos usados son diferentes (mayúsculas y espacios) por lo que se estandarizan para evitar confusion.

In [ ]:
char_file["ocupacion"].value_counts()

In [ ]:
char_file['ocupacion'] = char_file['ocupacion'].str.lower().str.replace(' ', '_')

**Otra ocupación**: es una columna con bastante diversidad en sus valores al tratarse de una pregunta abierta.

Se estandariza algunos valores, por ejemplo, aquellos donde las personas rellenaron con -, ., se pueden cambiar a declaro_sin_experiencia



In [ ]:
char_file["otra_ocupacion"].value_counts(dropna=False)

In [ ]:
char_file['otra_ocupacion'] = char_file['otra_ocupacion'].replace('no', 'declaro_sin_experiencia')
char_file['otra_ocupacion'] = char_file['otra_ocupacion'].replace('na', 'declaro_sin_experiencia')
char_file['otra_ocupacion'] = char_file['otra_ocupacion'].replace('no aplica', 'declaro_sin_experiencia')
char_file['otra_ocupacion'] = char_file['otra_ocupacion'].replace('no aplica.', 'declaro_sin_experiencia')

Todo lo que incluya la palabra estudiante o ning se cambiará a estudiante y NaN, respectivamente.

In [ ]:
char_file.loc[char_file['otra_ocupacion'].str.contains('estudiante', case=False, na=False), 'otra_ocupacion'] = 'estudiante'
char_file.loc[char_file['otra_ocupacion'].str.contains('ning', case=False, na=False), 'otra_ocupacion'] = 'campo_corregido'

También, cuando la columna ocupación y otra_ocupacion sean iguales, se dejará otra_ocupacion como "campo_corregido", para solamente tener en cuenta el valor de la ocupación

In [ ]:
char_file.loc[char_file['ocupacion'] == char_file['otra_ocupacion'], 'otra_ocupacion'] = 'campo_corregido'

In [ ]:
char_file["otra_ocupacion"].value_counts(dropna=False)

**Una alternativa para manejar esta columna es solamente aceptar valores cuando la columna ocupación indica el valor 'OTRO'.**

al hacer esto se perderian algunos detalles de otras ocupaciones que tienen las personas, pero se lograría estandarizar mejor las columnas.

Cuando la ocupación tenga un valor diferente a 'otro', se marca la otra_ocupacion como NA.

In [ ]:
char_file.loc[char_file['ocupacion'] != 'otro', 'otra_ocupacion'] = 'campo_corregido'

In [ ]:
columnas_interes = ['ocupacion', 'otra_ocupacion']
char_file.loc[char_file['otra_ocupacion'].notna(), columnas_interes]

Se evidencia que muchos llenaron la ocupacion como 'otro' pero en otra_ocupacion tienen categorias como 'estudiante'. Igualmente, muchos marcaron 'otro' para incluir la categoría de desempleo.

Por esto, cuando la ocupación sea otro y la otra_ocupacion se aestudiante, se deja en ocupación (el campo principal) que es estudiante, y en otra_ocupacion un "campo_corregido"

In [ ]:
condicion = (char_file['ocupacion'] == 'otro') & (char_file['otra_ocupacion'] == 'estudiante')

char_file.loc[condicion, 'ocupacion'] = 'estudiante'
char_file.loc[condicion, 'otra_ocupacion'] = 'campo_corregido'

In [ ]:
columnas_interes = ['ocupacion', 'otra_ocupacion']
char_file.loc[char_file['otra_ocupacion'].notna(), columnas_interes]

se estandariza para crear una sola categoria de otra_ocupacion de desempleado

In [ ]:
char_file.loc[char_file['otra_ocupacion'].str.contains('desemplead', case=False, na=False), 'otra_ocupacion'] = 'desempleado'

In [ ]:
char_file.loc[char_file['otra_ocupacion'].str.contains('deseempleada', case=False, na=False), 'otra_ocupacion'] = 'desempleado'

In [ ]:
columnas_interes = ['ocupacion', 'otra_ocupacion']
char_file.loc[char_file['otra_ocupacion'].notna(), columnas_interes]

Muchos pusieron como ocupación OTRO pero su otra ocupacion es la docencia, por lo que se categoriza como empleado y se marca otra_ocupacion como un campo corregido

In [ ]:
condicion = (char_file['ocupacion'] == 'otro') & (char_file['otra_ocupacion'] == 'docente')

char_file.loc[condicion, 'ocupacion'] = 'empleado'
char_file.loc[condicion, 'otra_ocupacion'] = 'campo_corregido'

In [ ]:
condicion = (char_file['ocupacion'] == 'otro') & (char_file['otra_ocupacion'] == 'docente de secundaria.')

char_file.loc[condicion, 'ocupacion'] = 'empleado'
char_file.loc[condicion, 'otra_ocupacion'] = 'campo_corregido'

In [ ]:
condicion = (char_file['ocupacion'] == 'otro') & (char_file['otra_ocupacion'] == 'profesor')

char_file.loc[condicion, 'ocupacion'] = 'empleado'
char_file.loc[condicion, 'otra_ocupacion'] = 'campo_corregido'

In [ ]:
condicion = (char_file['ocupacion'] == 'otro') & (char_file['otra_ocupacion'] == 'empleado')

char_file.loc[condicion, 'ocupacion'] = 'empleado'
char_file.loc[condicion, 'otra_ocupacion'] = 'campo_corregido'

los universitarios tambien se considerarán estudiantes.

In [ ]:
condicion = (char_file['ocupacion'] == 'otro') & (char_file['otra_ocupacion'] == 'universitario')

char_file.loc[condicion, 'ocupacion'] = 'estudiante'
char_file.loc[condicion, 'otra_ocupacion'] = 'campo_corregido'

In [ ]:
condicion = (char_file['ocupacion'] == 'otro') & (char_file['otra_ocupacion'] == 'ingeniero industrial en búsqueda activa de empleo')

char_file.loc[condicion, 'ocupacion'] = 'otro'
char_file.loc[condicion, 'otra_ocupacion'] = 'desempleado'

Ahora la columna de otra_ocupación cuenta con valores más estandarizados:

In [ ]:
(char_file["otra_ocupacion"].value_counts())

## dedicación semanal

se estandariza el texto

In [ ]:
char_file['dedicacionsemanal'] = char_file['dedicacionsemanal'].replace('7h_o_mas', '7 horas o más')

## nivel de programación

se estandariza reemplazando el texto para que quede uniforme.

In [ ]:
char_file['nivelprogramacion'] = char_file['nivelprogramacion'].str.lower()
char_file['nivelprogramacion'] = char_file['nivelprogramacion'].replace('básico', 'basico')
char_file['nivelprogramacion'] = char_file['nivelprogramacion'].replace('ninguno', 'nulo')


## lenguajes de programación y otro_lenguaje

Se realiza la estandarización de los valores, para que sean strings que son listas y poder ordenar y contar posteriormente.

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'No he programado antes', "['no_programa']"
)

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'Python', "['python']"
)

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'C/C++', "['c_c_plus']"
)

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'Java', "['java']"
)

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'Otro', "['otro']"
)

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'JavaScript', "['javascript']"
)

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'C#', "['c_sharp']"
)

In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].replace(
    'R', "['r']"
)

In [ ]:
def limpiar_y_ordenar(val):
    if isinstance(val, str) and val.startswith('['):
        try:
            val = ast.literal_eval(val)
        except:
            pass

    if isinstance(val, list):
        val = sorted([str(x) for x in val])

    return val

char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].apply(limpiar_y_ordenar)

In [ ]:
char_file['lenguajes_programacion'].value_counts()

In [ ]:
char_file['conteo_lenguajes'] = char_file['lenguajes_programacion'].str.len()

Se pueden observar algunos valores atipicos, porque hay una persona que marcó que conoce 8 lenguajes (incluida la opción de no programa).

In [ ]:
char_file.loc[char_file['conteo_lenguajes'] == 8, ['lenguajes_programacion', 'conteo_lenguajes']]

También, hay varias personas que marcaron la opción de no programa, junto con otra. Estos valores se tomarán como que no programa para evitar confusiones.

In [ ]:
condicion_error = (char_file['lenguajes_programacion'].apply(lambda x: 'no_programa' in x if isinstance(x, list) else False)) & \
                  (char_file['conteo_lenguajes'] > 1)

errores = char_file.loc[condicion_error, ['lenguajes_programacion', 'conteo_lenguajes']]

print(errores)

In [ ]:
def forzar_no_programa(lista):
    if isinstance(lista, list) and 'no_programa' in lista and len(lista) > 1:
        return ['no_programa']
    return lista

char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].apply(forzar_no_programa)


In [ ]:
char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
print(type(char_file['lenguajes_programacion'].iloc[0]))

Ahora se procede a observar la columna otro_lenguaje. Es una columna abierta por lo que se usa thefuzz para calcular la distancia de levenshtein para ayudar a la limpieza en cuanto a lenguajes de programación que hayan puesto en esa columna

In [ ]:

lenguajes_validos = ['python', 'javascript', 'c_sharp', 'java', 'ruby', 'c_c_plus']

def corregir_con_fuzzy(texto):
    if pd.isna(texto) or texto == '': return texto
    match, score = process.extractOne(texto, lenguajes_validos)
    return match if score >= 80 else texto

char_file['otro_lenguaje_clean'] = char_file['otro_lenguaje'].apply(corregir_con_fuzzy)

In [ ]:
print(f"Valores únicos originales: {char_file['otro_lenguaje'].nunique()}")
print(f"Valores únicos después: {char_file['otro_lenguaje_clean'].nunique()}")

In [ ]:
char_file['otro_lenguaje_clean'] = char_file['otro_lenguaje_clean'].str.lower()

In [ ]:
char_file[char_file['otro_lenguaje'].str.contains('python', case=False, na=False)]

In [ ]:
char_file.loc[char_file['otro_lenguaje_clean'].str.contains('ning', case=False, na=False), 'otro_lenguaje_clean'] = "declaro_sin_experiencia"

In [ ]:
char_file.loc[char_file['otro_lenguaje_clean'].str.contains('no ', case=False, na=False), 'otro_lenguaje_clean'] = "declaro_sin_experiencia"

In [ ]:
char_file['otro_lenguaje'].value_counts(dropna=False)

UPDATE ABRIL 17:
Dado que con esta columna vamos a hacer algo para el modelo de predicción, no vamos a reemplazar estos valores con NaN, sino con una categoria de "declaro_sin_experiencia", para diferenciar quién no respondió la encuesta, de quien dijo "no sé"

In [ ]:
valores_basura = ["na",
"no",
"No aplica",
"Ninguno",
"ninguno",
"No he programado antes",
"Na",
"No",
"no aplica",
"x",
"N.A",
"N/a",
"No Aplica",
"No aplica.",
"No he programado",
"No he programado antes.",
"Nunca he programado",
"n.a",
"n.a."
]

char_file['otro_lenguaje_clean'] = char_file['otro_lenguaje_clean'].str.strip()
char_file['otro_lenguaje_clean'] = char_file['otro_lenguaje_clean'].replace(valores_basura, "declaro_sin_experiencia")

In [ ]:
char_file['otro_lenguaje'].value_counts(dropna=False)

Ahora, se puede ver que aun en otro_lenguaje_clean hay valores que deberian pertenecer a lenguajes_programacion. Sin embargo, esta columna es un arreglo y ya puede contener el lenguaje que se especifico en otro_lenguaje_clean, por lo que se agrega el lenguaje de otro_lenguaje_clean a la columna de lenguajes_programacion y se elimina la etiqueta de 'otro'. Igualmente, otro_lenguaje_clean quedaria como nan para estos casos, ya que se movió la información de una columna a otra

In [ ]:
char_file[char_file['otro_lenguaje_clean'] == 'python'][['lenguajes_programacion', 'otro_lenguaje','otro_lenguaje_clean']]

Update 17 de abril: no se va a dejar como Nan el otro_lenguaje, para marcar que la persona sí contestó pero fue algo duplicado

In [ ]:
lenguajes_objetivo = ['python', 'javascript', 'c_sharp', 'java', 'ruby', 'c_c_plus']

def literal_eval_safe(x):
    try:
        if isinstance(x, str) and x.startswith('['):
            return ast.literal_eval(x)
        return x if isinstance(x, list) else []
    except:
        return []

char_file['lenguajes_programacion'] = char_file['lenguajes_programacion'].apply(literal_eval_safe)

def integrar_lenguajes(row):
    lista = row['lenguajes_programacion']
    otro = str(row['otro_lenguaje_clean']).lower().strip()

    if otro in lenguajes_objetivo:
        s = set(lista)
        s.add(otro)
        s.discard('otro')
        return list(s)

    return lista

mask = char_file['otro_lenguaje_clean'].isin(lenguajes_objetivo)
char_file.loc[mask, 'lenguajes_programacion'] = char_file[mask].apply(integrar_lenguajes, axis=1)

char_file.loc[mask, 'otro_lenguaje_clean'] = "campo_corregido"

In [ ]:
datos_rescatados = (char_file['otro_lenguaje_clean'] == "campo_corregido").sum()

print(f"Se rescataron {datos_rescatados} registros de 'otro_lenguaje_clean'.")

In [ ]:
df_python = char_file[char_file['lenguajes_programacion'].apply(lambda x: 'otro' in x if isinstance(x, list) else False)]

df_python[['lenguajes_programacion', 'otro_lenguaje_clean']]

In [ ]:
char_file.drop(columns=['otro_lenguaje'], inplace=True)
char_file.rename(columns={'otro_lenguaje_clean': 'otro_lenguaje'}, inplace=True)

## dónde conoció el curso y otro_medio

Se realiza limpieza y estandarizacion de los valores

In [ ]:
char_file['dondeconocioelcurso'] = char_file['dondeconocioelcurso'].str.lower().replace(' ', '_')
char_file['dondeconocioelcurso'] = char_file['dondeconocioelcurso'].replace('recomendación personal', 'recomendacion')
char_file['dondeconocioelcurso'] = char_file['dondeconocioelcurso'].replace('correo electrónico', 'email')
char_file['dondeconocioelcurso'] = char_file['dondeconocioelcurso'].replace('página web', 'pagina web')

Por medio de fuzzy se empieza la limpieza del campo otro_medio. Se reemplaza por los medios válidos lo que los contenga en el campo abierto

In [ ]:

medios_validos = ['facebook', 'recomendacion', 'google', 'pagina web', 'email', 'instagram',
                  'linkedin', 'youtube', 'sms']

def corregir_con_fuzzy(texto):
    if pd.isna(texto) or texto == '': return texto
    match, score = process.extractOne(texto, medios_validos)
    return match if score >= 80 else texto

char_file['otro_medio_clean'] = char_file['otro_medio'].apply(corregir_con_fuzzy)

Se elimina valores basura comunes en esa columna

In [ ]:
valores_basura = ["na",
"no",
"No aplica",
"Ninguno",
"ninguno",
"No he programado antes",
"Na",
"No",
"no aplica",
"x",
"N.A",
"N/a",
"No Aplica",
"No aplica.",
"No he programado",
"No he programado antes.",
"Nunca he programado"
]

char_file['otro_medio_clean'] = char_file['otro_medio_clean'].replace(valores_basura, "declaro_sin_experiencia")

También, luego de la limpieza de fuzzy, cuando la columna dónde conoció el curso y otra_medio sean iguales, se dejará otro_medio como NA, para solamente tener en cuenta el valor de la ocupación

In [ ]:
char_file.loc[char_file['dondeconocioelcurso'] == char_file['otro_medio_clean'], 'otro_medio_clean'] = "campo_corregido"

También, si dondeconocioelcurso es NA, pero otro_medio_clean ahora tiene un valor dentro de la lista de medios validos, intercambiamos los valores.

In [ ]:
medios_validos = ['facebook', 'recomendacion', 'google', 'pagina web', 'email', 'instagram',
                  'linkedin', 'youtube', 'sms']

condicion = (char_file['dondeconocioelcurso'].isna()) & (char_file['otro_medio_clean'].isin(medios_validos))

char_file.loc[condicion, 'dondeconocioelcurso'] = char_file.loc[condicion, 'otro_medio_clean']

char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

Se realiza una estandarización de otro_medio_clean con el correo electrónico y las recomendaciones. Una vez más, si el campo de dondeconocioelcurso está vacio pero otro_medio tiene datos, se intercambian. Si son iguales, otro_medio será NA.
También, si las personas marcaron OTRO, pero el otro_medio es recomendación, se mueve la data a la columna principal

In [ ]:
char_file.loc[char_file['otro_medio_clean'].str.contains('correo elec', case=False, na=False), 'otro_medio_clean'] = 'email'


In [ ]:
char_file.loc[char_file['otro_medio_clean'].str.contains('correo', case=False, na=False), 'otro_medio_clean'] = 'email'


In [ ]:
char_file.loc[char_file['otro_medio_clean'].str.contains('recom', case=False, na=False), 'otro_medio_clean'] = 'recomendacion'


In [ ]:
char_file.loc[char_file['otro_medio_clean'].str.contains('reco', case=False, na=False), 'otro_medio_clean'] = 'recomendacion'


In [ ]:
condicion = (char_file['dondeconocioelcurso'] == 'otro') & (char_file['otro_medio_clean'] == 'recomendacion')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'recomendacion'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
condicion = (char_file['dondeconocioelcurso'].isna()) & (char_file['otro_medio_clean'] == 'recomendacion')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'recomendacion'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
condicion = (char_file['dondeconocioelcurso'] == 'recomendacion') & (char_file['otro_medio_clean'] == 'recomendacion')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'recomendacion'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
condicion = (char_file['dondeconocioelcurso'] == 'email') & (char_file['otro_medio_clean'] == 'email')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'email'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
condicion = (char_file['dondeconocioelcurso'].isna()) & (char_file['otro_medio_clean'] == 'email')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'email'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
char_file.loc[char_file['otro_medio_clean'].str.contains('web', case=False, na=False), 'otro_medio_clean'] = 'pagina web'


In [ ]:
condicion = (char_file['dondeconocioelcurso'] == 'pagina web') & (char_file['otro_medio_clean'] == 'pagina web')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'pagina web'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
condicion = (char_file['dondeconocioelcurso'] == 'otro') & (char_file['otro_medio_clean'] == 'pagina web')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'pagina web'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
condicion = (char_file['dondeconocioelcurso'] == 'recomendacion') & (char_file['otro_medio_clean'] == 'pagina web')

char_file.loc[condicion, 'dondeconocioelcurso'] = 'recomendacion'
char_file.loc[condicion, 'otro_medio_clean'] = "campo_corregido"

In [ ]:
char_file.drop(columns=['otro_medio'], inplace=True)
char_file.rename(columns={'otro_medio_clean': 'otro_medio'}, inplace=True)

## motivación y otra motivación

In [ ]:
(char_file["motivacion"].value_counts())

In [ ]:
char_file['motivacion'] = char_file['motivacion'].str.lower().replace(' ', '_')
char_file['motivacion'] = char_file['motivacion'].replace('ml_ds', 'quiero entrar al mundo de Machine learning y Ciencia de datos')
char_file['motivacion'] = char_file['motivacion'].replace('quiero entrar al mundo de machine learning y ciencia de datos', 'quiero entrar al mundo de Machine learning y Ciencia de datos')
char_file['motivacion'] = char_file['motivacion'].replace('primer_lenguaje', 'quiero aprender a programar computadores en general (es mi primer lenguaje de programación)')
char_file['motivacion'] = char_file['motivacion'].replace('desarrollo_backend', 'quiero saber acerca de Desarrollo backend')
char_file['motivacion'] = char_file['motivacion'].replace('otra', 'otro')
char_file['motivacion'] = char_file['motivacion'].replace('desarrollo_web', 'quiero involucrarme en Desarrollo web')
char_file['motivacion'] = char_file['motivacion'].replace('answer text missing', np.nan)
char_file['motivacion'] = char_file['motivacion'].str.lower()

In [ ]:
valores_basura = ["na",
"no",
"No aplica",
"Ninguno",
"ninguno",
"No he programado antes",
"Na",
"No",
"no aplica",
"x",
"N.A",
"N/a",
"No Aplica",
"No aplica.",
"No he programado",
"No he programado antes.",
"Nunca he programado",
"na"
]

char_file['otra_motivacion'] = char_file['otra_motivacion'].replace(valores_basura, "declaro_sin_experiencia")

In [ ]:
char_file['otra_motivacion'].value_counts()

## razón para realizar el curso

No se estandariza el texto, se analizará más adelante

## archivo final de caracterización estandarizado

In [ ]:
char_file.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-characterization-file.csv', index=False) # index=False prevents writing the index as a separate column

# 2.Autoevaluación


In [ ]:
eval_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-self-evaluation-file-completed.csv'


In [ ]:
eval_file = pd.read_csv(eval_file_path)


In [ ]:
len(eval_file)

In [ ]:
eval_file.columns

In [ ]:
questions_dict = {
    "pregunta1" : "Mejora en programacion",
    "pregunta2" : "Notebooks de Python",
    "pregunta3" : "Pruebas de código",
    "pregunta4" : "Retroalimentación",
    "pregunta5" : "Abstraer un problema en forma de algoritmo e implementarlo en un programa con el lenguaje de programación Python.",
    "pregunta6" : "Conocer y aplicar los principios de manejo de variables, tipos de datos y operaciones entre valores numéricos y de texto en Python.",
    "pregunta7" : "Diseñar e implementar algoritmos con decisiones basadas en la evaluación de expresiones booleanas.",
    "pregunta8" : "Diseñar e implementar algoritmos con la aplicación de ciclos iterativos.",
    "pregunta9" : "Distinguir y elegir la colección apropiada para la resolución de un problema con una estructura de datos compleja.",
    "pregunta10" : "Usar colecciones en el diseño e implementación de algoritmos para la solución de problemas.",
    "pregunta11" : "Uso de pistas",
    "pregunta12" : "Utilidad uncode",
    "pregunta13" : "Videos",
    "pregunta14" : "Visualización de código",
    "pregunta15" : "¿Cuál considera que es la mayor fortaleza de este módulo que le ayudó a alcanzar los objetivos de aprendizaje?",
    "pregunta16" : "¿Qué aspectos considera que podemos mejorar en este curso para que le permita alcanzar en mayor medida los objetivos de aprendizaje?",
    "pregunta17" : "Declarar y manipular listas, tuplas, matrices, diccionarios y conjuntos con el lenguaje de programación Python.",
    "pregunta18" : "Foro",
    "pregunta19" : "Dedicacion semanal"
}

In [ ]:
questions_type_dict = {
    "pregunta1" : "one",
    "pregunta2" : "one",
    "pregunta3" : "one",
    "pregunta4" : "one",
    "pregunta5" : "two",
    "pregunta6" : "two",
    "pregunta7" : "two",
    "pregunta8" : "two",
    "pregunta9" : "two",
    "pregunta10" : "two",
    "pregunta11" : "one",
    "pregunta12" : "one",
    "pregunta13" : "one",
    "pregunta14" : "one",
    "pregunta15" : "skip",
    "pregunta16" : "skip",
    "pregunta17" : "two",
    "pregunta18" : "one",
    "pregunta19" : "skip"
}


## Limpieza de columnas preguntas cerradas

se estandarizan los valores de las preguntas cerradas ya que varían según la plataforma.

In [ ]:
def clean_self_eval_cols(col):

  eval_file[col] = eval_file[col].str.strip()
  eval_file[col] = eval_file[col].str.lower()

  question_type = questions_type_dict.get(col)
  if question_type == 'one':
    eval_file[col] = eval_file[col].replace('6. totalmente de acuerdo', '6-totalmente de acuerdo')
    eval_file[col] = eval_file[col].replace('5. de acuerdo', '5-de acuerdo')
    eval_file[col] = eval_file[col].replace('4. algo de acuerdo', '4-algo de acuerdo')
    eval_file[col] = eval_file[col].replace('3. algo en desacuerdo', '3-algo en desacuerdo')
    eval_file[col] = eval_file[col].replace('2. en desacuerdo', '2-en desacuerdo')
    eval_file[col] = eval_file[col].replace('1. totalmente en desacuerdo', '1-totalmente en desacuerdo')
  elif question_type == 'two':
    eval_file[col] = eval_file[col].replace('2. bajo', '2-bajo')
    eval_file[col] = eval_file[col].replace('3. medio', '3-medio')
    eval_file[col] = eval_file[col].replace('4. alto', '4-alto')
    eval_file[col] = eval_file[col].replace('5. muy alto', '5-muy alto')
  else:
    print("Skip")

In [ ]:
def main_clean_eval_cols():
  col_name = "pregunta"
  for i in range(1, 20):
    full_col_name = col_name + str(i)
    question_type = questions_type_dict.get(str(full_col_name))
    if question_type != "skip":
      clean_self_eval_cols(full_col_name)

In [ ]:
main_clean_eval_cols()

## archivo final de autoevaluación estandarizado

In [ ]:
eval_file.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-evaluation-file.csv', index=False) # index=False prevents writing the index as a separate column

# 3.Submissions

In [ ]:
sub_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-submissions-file.csv'


In [ ]:
sub_file = pd.read_csv(sub_file_path)


In [ ]:
sub_file.columns

In [ ]:
len(sub_file)

Hay 176736 submissions para ambas plataformas

In [ ]:
sub_file[sub_file['status'] == 'done'][['grade', 'status', 'result', 'platform']]

In [ ]:
sub_file['result'].value_counts()

In [ ]:
sub_file['url'].value_counts(dropna=False)

In [ ]:
sub_file[['_id']].head(10)

In [ ]:
sub_file['outcome_consumer_key'].head(10)

## Descripción de columnas:

- '_id': identificador alfanumerico de la entrega
- 'archive': identificador alfanumerico del archivo de python donde está el código correspondiente a la entrega.
- 'best': booleano que indica si esta entrega tiene la calificación más alta. Si las entregas tienen el mismo valor, toma el más reciente.
- 'courseid': identificador del curso al que pertenece la entrega.
- 'custom[custom_additional_info]': json que indica la respuesta del calificador.
- 'custom[custom_summary_result]': indica si la entrega fue aceptada (accepted) o tiene una respuesta incorrecta (wrong answer).
- 'grade': nota asignada a la entrega según el calificador.
- 'input'
- 'is_late_submission': booleano que indica si la tarea se envío después de la fecha indicada de entrega.
- 'outcome_consumer_key'
- 'outcome_result_id'
- 'outcome_service_url'
- 'penalty': penalidad por sobrepasar el número de intentos. Es 0 para todos en este caso.
- 'response_type': json para todos.
- 'result': indica si el código fue ejecutado  exitosamente o falló, y por qué falló.
- 'status': indica si la entrega está hecha o no (hay status en done, pero que el resultado es failed),
- 'stderr': todos son NaN
- 'stdout': todos son NaN
- 'submitted_on': fecha y hora de la entrega.
- 'taskid': identificador alfanumérico de la actividad.
- 'text': texto en html que muestra ayuda para ejecutar la tarea
- 'url': url del submission, usando el _id de la entrega
- 'username': usuario
- 'platform': plataforma (edunext o openedx)
- 'archivo_origen': ruta al archivo de metadata al que pertenece la entrega.
- 'actividad': actividad a la que pertenece la entrega

Se eliminan las columnas que tienen el mismo valor en todos sus rows y que no se considera aportarán algo a la investigación:

In [ ]:
sub_file.drop(columns=['penalty'], inplace=True)
sub_file.drop(columns=['response_type'], inplace=True)
sub_file.drop(columns=['stderr'], inplace=True)
sub_file.drop(columns=['stdout'], inplace=True)

## best: mejor entrega

In [ ]:
sub_file[['best']].value_counts()

## courseid

solo hay dos valores para esta variable, y realmente no refleja a qué curso corresponde la entrega. Para esto se puede usar la fecha de entrega. Por esto, también se quita esta columna.

In [ ]:
sub_file[['courseid', 'submitted_on']]

In [ ]:
sub_file['courseid'].value_counts(dropna=False)

In [ ]:
sub_file.drop(columns=['courseid'], inplace=True)

## submitted_on

se convierte a una fecha válida para Pandas la columna solo_fecha

In [ ]:
sub_file['submitted_on'] = sub_file['submitted_on'].str[:19]
sub_file['submitted_on'] = pd.to_datetime(sub_file['submitted_on'])

sub_file['solo_fecha'] = sub_file['submitted_on'].dt.date

In [ ]:
sub_file['solo_fecha'] = pd.to_datetime(sub_file['solo_fecha'])

se crean columnas de año y mes

In [ ]:
sub_file['ano'] = sub_file['solo_fecha'].dt.year
sub_file['mes'] = sub_file['solo_fecha'].dt.month


## actividad

como existe una columna para identifica de qué plataforma la actividad, se procede a estandarizar las tareas para que sean consistentes entre plataforma.

In [ ]:
sub_file['actividad'].value_counts().sort_index()

In [ ]:
sub_file['actividad'] = sub_file['actividad'].str.replace('edunext_', '')

In [ ]:
sub_file['actividad'] = sub_file['actividad'].str.replace('actividad_', 'a')

In [ ]:
sub_file['actividad'] = sub_file['actividad'].str.replace('a6_cursores_y_áreas', 'a6_cursores_y_areas')

In [ ]:
sub_file['actividad'] = sub_file['actividad'].str.replace('a7_años_bisisestos', 'a7_bisisestos')

In [ ]:
sub_file['actividad'] = sub_file['actividad'].str.replace('a6a_cursores_y_areas', 'a6_cursores_y_areas')

In [ ]:
sub_file['actividad'] = sub_file['actividad'].str.replace('a7a_bisisestos', 'a7_bisisestos')

In [ ]:
sub_file['actividad'] = sub_file['actividad'].str.replace('a3_operadores_numéricos', 'a3_operadores_numericos')

In [ ]:
sub_file[sub_file['actividad'] == 'a4_ejemplo_1'][['actividad', 'platform']]

In [ ]:
sub_file['actividad'].value_counts().sort_index()

Se unifican las siguientes tareas:

- a6_cursores_y_areas
- a6a_cursores_y_areas
- a7_bisisestos
- a7a_bisisestos


## usuario en los submissions

se limpia el string del usuario

In [ ]:
sub_file['username'] = sub_file['username'].str.lower()
sub_file['username'] = sub_file['username'].str.replace('[', '')
sub_file['username'] = sub_file['username'].str.replace(']', '')
sub_file['username'] = sub_file['username'].str.replace("'", '')

## archivo final de submissions estandarizado

In [ ]:
sub_file.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-submissions-file.csv', index=False) # index=False prevents writing the index as a separate column

# 4.COMPENDIO CARACTERIZACIÓN - SUBMISSIONS

In [ ]:
submissions_user_counts = sub_file.groupby(['username', 'platform']).size().reset_index(name='count')

In [ ]:
submissions_char = pd.merge(
    submissions_user_counts,
    char_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=True
)


Hay 463 usuarios que tienen submissions y están en el archivo de caracterización. Habrá un total de 18322 para analizar para este grupo.

UPDATE ABRIL 14: ahora son 1070 para un total de 40567

In [ ]:
len(submissions_char)

In [ ]:
total = submissions_char['count'].sum()
print(f"La suma total es: {total}")

In [ ]:
total_submissions_char = pd.merge(
    submissions_char,
    sub_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_submissions_char))

In [ ]:
total_submissions_char.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-char-submissions-file.csv', index=False) # index=False prevents writing the index as a separate column

# 5.COMPENDIO SUBMISSIONS - AUTOEVALUACIÓN

In [ ]:
submissions_user_counts = sub_file.groupby(['username', 'platform']).size().reset_index(name='count')

In [ ]:
submissions_eval = pd.merge(
    submissions_user_counts,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=True
)


307 - 19275

Update abril 14 : 724 -  46545

In [ ]:
len(submissions_eval)

In [ ]:
total = submissions_eval['count'].sum()
print(f"La suma total es: {total}")

In [ ]:
total_submissions_eval = pd.merge(
    submissions_eval,
    sub_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_submissions_eval))

In [ ]:
total_submissions_eval.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-eval-submissions-file.csv', index=False) # index=False prevents writing the index as a separate column

# 6.Compendio final: Submissions, encuesta de caracterización y autoevaluación

Se utiliza el submissions_char creado previamente para unirlo al archivo de autoevaluacion

In [ ]:
submissions_char.columns

In [ ]:
submissions_char_eval = pd.merge(
    submissions_char,
    eval_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(submissions_char_eval))

In [ ]:
total = submissions_char_eval['count'].sum()
print(f"La suma total es: {total}")

In [ ]:
submissions_char_eval.head()

Por ahora, son 190 usuarios los que poseen la traza completa de su información, para un total de 12203 submissions a analizar.

Update abril 14: 339 con 22554




In [ ]:
submissions_char_eval.columns

In [ ]:
submissions_char_eval.drop(columns=['email_y', 'name_y', 'uncode_usr_y'], inplace=True)

Esta seria la información total que tendriamos por usuario:

In [ ]:
total_submissions_char_eval = pd.merge(
    submissions_char_eval,
    sub_file,
    how='inner',
    left_on=['username', 'platform'],
    right_on=['username', 'platform'],
    indicator=False
)

print("match:", len(total_submissions_char_eval))

In [ ]:
total_submissions_char_eval.columns

In [ ]:
total_submissions_char_eval

In [ ]:
total_submissions_char_eval.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-total-char-submissions-eval-file.csv', index=False) # index=False prevents writing the index as a separate column